In [1]:
# 📌 Notebook: notebooks/embedding_model_demo.ipynb

# 📦 Imports
import pandas as pd
import random
import sys
import os

# 🔧 Set up path for src
sys.path.append(os.path.abspath('../src'))

# 🧠 Load the model
from embedding_model import EmbeddingRecommender

# 📁 Load Data
df = pd.read_pickle('../data/processed_playlists.pkl')  # from EDA step
df_tracks = pd.read_pickle('../data/processed_tracks.pkl')  # track metadata

# 🧱 Initialize and Train the Embedding Model
embedding_rec = EmbeddingRecommender(vector_size=64, window=5, min_count=1)

# 🧹 Prepare training data (list of playlists as lists of track URIs)
playlists = embedding_rec.prepare_training_data(df)

# 🚀 Train Word2Vec
embedding_rec.train(playlists)

# 🎧 Pick a random track URI from a random playlist
random_playlist = random.choice(playlists)
random_track = random.choice(random_playlist)
print("🎵 Sample Track URI:", random_track)

# 🤖 Recommend similar tracks
recommended_uris = embedding_rec.recommend_similar_tracks(random_track, top_n=10)
print("🎯 Recommended Track URIs:")
print(recommended_uris)

# 🧾 Optionally show metadata
display(df_tracks[df_tracks['track_uri'].isin(recommended_uris)][['track_name', 'artist_name']])


🎵 Sample Track URI: spotify:track:645IXl7jWVyMVKYCSJWq2N
🎯 Recommended Track URIs:
['spotify:track:22SoE5ihFLcNPELv9qEYXS', 'spotify:track:2NczH1flzBaqOKE4ICkgoc', 'spotify:track:1HDihF11CPmyYdvXcckndz', 'spotify:track:5Fm0370X5xpSsDy2AL0Zra', 'spotify:track:3gJnvWoh2DZITfB1z32OcK', 'spotify:track:6zFisC4ocYM8248gwnHrnd', 'spotify:track:7LaMVxZCZoX432x8fAt6ts', 'spotify:track:46frEscYLoQ483ZjvmlmbE', 'spotify:track:78mHFsRy8VkMC2Kilw2vhA', 'spotify:track:6cVcfeFrFyubyrKhp2eIXW']


,track_name,artist_name
36087,Breathe A•gain,Couros
42403,Not Me (feat. Two Feet),Melvv
169465,Stay The Night,SCARFɆ
191176,Faded On Your Love,Griffin Stoller
248222,Junkie (feat. Nevve & Monstre),Kill Paris
299498,Girl,Moglii
571181,Like It's Over (feat. MNDR) [Howle Remix],Jai Wolf
574567,Back to Start (feat. Dylan Dunlap),Stonebank
671734,Love,Tilka
1667357,Handcrafted,Dropout


In [4]:
track_uri = "spotify:track:22SoE5ihFLcNPELv9qEYXS"
print("Similar tracks to:", track_uri)
print(embedding_rec.recommend_similar_tracks(track_uri, top_n=5))


Similar tracks to: spotify:track:22SoE5ihFLcNPELv9qEYXS
['spotify:track:602BJRfliP0zKFbLaE7HvV', 'spotify:track:478RXItI4mOZELDyUXMJYR', 'spotify:track:1HDihF11CPmyYdvXcckndz', 'spotify:track:0S9EgNovG2nLubzErxZVok', 'spotify:track:0Rk02wkJnsXj6L5UnzWUBX']


In [5]:
def evaluate_embedding_model(embedding_rec, df, k=10):
    hit_count = 0
    total = 0

    for _, group in df.groupby('playlist_id'):
        if len(group) < 5:
            continue
        playlist_uris = list(group['track_uri'])
        test_track = playlist_uris[-1]
        input_tracks = playlist_uris[:-1]

        recommendations = embedding_rec.recommend_for_playlist(input_tracks, top_n=k)
        if test_track in recommendations:
            hit_count += 1
        total += 1

    recall_at_k = hit_count / total if total > 0 else 0
    print(f"🎯 Recall@{k}: {recall_at_k:.4f}")


In [6]:
evaluate_embedding_model(embedding_rec, df)


🎯 Recall@10: 0.0085


In [8]:
# 💾 Save model
import os
import pickle

# 💾 Save the trained Word2Vec model
model_save_path = '../data/models/embedding_model.pkl'
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)

with open(model_save_path, 'wb') as f:
    pickle.dump(embedding_rec.model, f)

print(f"✅ Embedding model saved to: {model_save_path}")



✅ Embedding model saved to: ../data/models/embedding_model.pkl
